In [ ]:
# =============================================================
# CASO 5 – E-commerce MercadoFresco
# Efectividad de Campaña de Marketing
# Modelo: Random Forest 
# =============================================================
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Estilo global de gráficas
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})
PALETTE = ["#1D9E75", "#D85A30", "#378ADD", "#BA7517", "#993556"]

In [ ]:
# 1. CARGA Y EXPLORACIÓN INICIAL 
df = pd.read_csv("caso5_campana_marketing.csv")
 
print("=" * 55)
print("  RESUMEN DEL DATASET")
print("=" * 55)
print(f"  Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
print(f"  Valores nulos: {df.isnull().sum().sum()}")
print()
 
# Distribución de la variable objetivo
conv = df["realizo_compra"].value_counts()
conv_pct = df["realizo_compra"].value_counts(normalize=True) * 100
print("  Distribución variable objetivo:")
print(f"    No compró (0): {conv[0]}  ({conv_pct[0]:.1f}%)")
print(f"    Sí compró (1): {conv[1]}  ({conv_pct[1]:.1f}%)")
print()
print(df.describe().T.round(2).to_string())

In [ ]:
# 2. EDA – ANÁLISIS EXPLORATORIO 
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("MercadoFresco – Análisis exploratorio de datos (EDA)", fontsize=15, fontweight="bold", y=1.01)

# 2.1 Distribución variable objetivo
ax = axes[0, 0]
bars = ax.bar(["No compró (0)", "Sí compró (1)"], conv.values,
              color=["#D85A30", "#1D9E75"], width=0.5, edgecolor="white")
for bar, val, pct in zip(bars, conv.values, conv_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"{val}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=11)
ax.set_title("Distribución de conversión")
ax.set_ylabel("Cantidad de clientes")
ax.set_ylim(0, conv.max() * 1.2)

# 2.2 Tasa de conversión por canal
ax = axes[0, 1]
canal_conv = df.groupby("canal_marketing")["realizo_compra"].mean().sort_values(ascending=False) * 100
bars = ax.bar(canal_conv.index, canal_conv.values, color=PALETTE[:len(canal_conv)], width=0.6)
for bar, val in zip(bars, canal_conv.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_title("Tasa de conversión por canal")
ax.set_ylabel("Tasa de conversión (%)")
ax.set_ylim(0, canal_conv.max() * 1.25)
ax.set_xticklabels(canal_conv.index, rotation=20, ha="right")

# 2.3 Tasa de conversión por hora de envío
ax = axes[0, 2]
orden_hora = ["Manana", "Mediodia", "Tarde", "Noche"]
hora_conv = (df.groupby("hora_envio")["realizo_compra"].mean() * 100).reindex(orden_hora)
bars = ax.bar(hora_conv.index, hora_conv.values, color=PALETTE[:4], width=0.6)
for bar, val in zip(bars, hora_conv.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_title("Tasa de conversión por hora")
ax.set_ylabel("Tasa de conversión (%)")
ax.set_ylim(0, hora_conv.max() * 1.25)

# 2.4 Distribución edad por grupo
ax = axes[1, 0]
ax.hist(df[df["realizo_compra"] == 0]["edad_cliente"], bins=18, alpha=0.7,
        color="#D85A30", label="No compró", edgecolor="white")
ax.hist(df[df["realizo_compra"] == 1]["edad_cliente"], bins=18, alpha=0.7,
        color="#1D9E75", label="Sí compró", edgecolor="white")
ax.set_title("Distribución de edad por conversión")
ax.set_xlabel("Edad del cliente")
ax.set_ylabel("Frecuencia")
ax.legend()

# 2.5 Compras previas vs conversión
ax = axes[1, 1]
ax.boxplot(
    [df[df["realizo_compra"] == 0]["compras_previas"],
     df[df["realizo_compra"] == 1]["compras_previas"]],
    labels=["No compró", "Sí compró"],
    patch_artist=True,
    boxprops=dict(facecolor="#E1F5EE", color="#1D9E75"),
    medianprops=dict(color="#085041", linewidth=2),
    whiskerprops=dict(color="#0F6E56"),
    capprops=dict(color="#0F6E56"),
    flierprops=dict(marker="o", markerfacecolor="#1D9E75", markersize=4, alpha=0.5)
)
ax.set_title("Compras previas vs conversión")
ax.set_ylabel("Número de compras previas")
 
# 2.6 Heatmap canal × hora (tasa de conversión)
ax = axes[1, 2]
pivot = df.pivot_table(values="realizo_compra", index="canal_marketing",
                       columns="hora_envio", aggfunc="mean") * 100
pivot = pivot.reindex(columns=orden_hora)
sns.heatmap(pivot, ax=ax, annot=True, fmt=".1f", cmap="YlGn",
            linewidths=0.5, cbar_kws={"label": "% conversión"})
ax.set_title("Conversión (%) – Canal × Hora")
ax.set_xlabel("Hora de envío")
ax.set_ylabel("Canal")
 
plt.tight_layout()
plt.savefig("eda_mercadofresco.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n[✓] Gráfica EDA guardada: eda_mercadofresco.png")

In [ ]:
# 3. PREPROCESAMIENTO 

print("\n" + "=" * 55)
print("  PREPROCESAMIENTO")
print("=" * 55)
 
df_model = df.copy()
 
# Variables categóricas a codificar
cat_cols = ["genero", "rango_ingreso", "canal_marketing",
            "dia_envio", "hora_envio", "categoria_producto"]
 
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")
 
# Variables numéricas — Random Forest no requiere escalado
# pero se documenta el rango para el informe
num_cols = ["edad_cliente", "compras_previas", "dias_inactivo", "descuento_porcentaje"]
print(f"\n  Variables numéricas (sin escalado): {num_cols}")
print("  Random Forest no requiere normalización — basado en particiones de árbol")
 
X = df_model.drop("realizo_compra", axis=1)
y = df_model["realizo_compra"]
 
print(f"\n  X shape: {X.shape}  |  y distribución: {dict(y.value_counts())}")

In [ ]:
# 4. TRAIN / TEST SPLIT 80/20 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print("\n" + "=" * 55)
print("  SPLIT 80/20 (estratificado)")
print("=" * 55)
print(f"  Train: {X_train.shape[0]} registros")
print(f"  Test:  {X_test.shape[0]} registros")
print(f"  Conversión en train: {y_train.mean()*100:.1f}%")
print(f"  Conversión en test:  {y_test.mean()*100:.1f}%")

In [ ]:
# 5. ENTRENAMIENTO RANDOM FOREST 

print("\n" + "=" * 55)
print("  ENTRENAMIENTO – Random Forest Classifier")
print("=" * 55)
 
rf = RandomForestClassifier(
    n_estimators=200,        # 200 árboles — buen balance velocidad/estabilidad
    max_depth=12,            # profundidad controlada para evitar overfitting
    min_samples_split=10,    # mínimo de muestras para dividir un nodo
    min_samples_leaf=5,      # mínimo de muestras en una hoja
    class_weight="balanced", # compensa desbalance 66%/34%
    random_state=42,
    n_jobs=-1                # usa todos los núcleos disponibles
)
rf.fit(X_train, y_train)
print("  Modelo entrenado con 200 árboles, class_weight='balanced'")
 
# Validación cruzada 5-fold sobre train
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
print(f"\n  Validación cruzada 5-fold (AUC-ROC):")
print(f"  Scores: {cv_scores.round(3)}")
print(f"  Media: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

In [ ]:
# 6. EVALUACIÓN Y MÉTRICAS 
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
 
auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
 
print("\n" + "=" * 55)
print("  MÉTRICAS DE EVALUACIÓN (Test set)")
print("=" * 55)
print(f"\n  AUC-ROC : {auc:.4f}   ← discriminación global")
print(f"  F1-Score: {f1:.4f}   ← balance precisión/recall")
print()
print(classification_report(y_test, y_pred,
      target_names=["No compró (0)", "Sí compró (1)"]))

In [ ]:
# 7. GRÁFICAS DE EVALUACIÓN 
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("MercadoFresco – Evaluación del modelo Random Forest",
             fontsize=14, fontweight="bold")
 
# 7.1 Curva ROC
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax.plot(fpr, tpr, color="#1D9E75", linewidth=2.5, label=f"AUC = {auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Clasificador aleatorio")
ax.fill_between(fpr, tpr, alpha=0.08, color="#1D9E75")
ax.set_xlabel("Tasa de Falsos Positivos")
ax.set_ylabel("Tasa de Verdaderos Positivos")
ax.set_title("Curva ROC")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
 
# 7.2 Matriz de confusión
ax = axes[1]
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["No compró", "Sí compró"])
disp.plot(ax=ax, colorbar=False, cmap="YlGn")
ax.set_title("Matriz de confusión")
ax.grid(False)
 
# 7.3 Feature Importance
ax = axes[2]
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
colors = ["#1D9E75" if v >= feat_imp.quantile(0.7) else "#9FE1CB" for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=colors, edgecolor="white")
ax.set_title("Importancia de variables (Feature Importance)")
ax.set_xlabel("Importancia relativa")
for i, (idx, val) in enumerate(feat_imp.items()):
    ax.text(val + 0.002, i, f"{val:.3f}", va="center", fontsize=9)
ax.set_xlim(0, feat_imp.max() * 1.2)
 
plt.tight_layout()
plt.savefig("evaluacion_modelo.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n[✓] Gráfica de evaluación guardada: evaluacion_modelo.png")

In [ ]:
# 8. INTERPRETACIÓN CON LENGUAJE DE NEGOCIO 
print("\n" + "=" * 55)
print("  INTERPRETACIÓN DE RESULTADOS – LENGUAJE DE NEGOCIO")
print("=" * 55)
 
feat_imp_sorted = feat_imp.sort_values(ascending=False)
top3 = feat_imp_sorted.head(3)
 
print(f"""
  RESUMEN EJECUTIVO PARA EL EQUIPO DE MARKETING:
 
  El modelo predice con {auc*100:.1f}% de capacidad discriminatoria
  (AUC-ROC) quién comprará tras recibir una campaña.
 
  Las 3 variables que más influyen en la decisión de compra son:
""")
for i, (var, imp) in enumerate(top3.items(), 1):
    print(f"    {i}. {var:25s} → {imp*100:.1f}% de importancia")
 
print(f"""
  IMPLICACIONES OPERATIVAS:
 
  1. Segmentación prioritaria: enfocar el {conv_pct[1]:.0f}% de presupuesto
     en clientes con perfil predicho = Sí compró.
 
  2. Canal × Hora óptimos: ver heatmap EDA —
     la combinación de mayor conversión identifica la "ventana dorada"
     donde cada peso invertido rinde más.
 
  3. Umbral de decisión ajustable:
     - Si el costo del envío es BAJO  → bajar umbral (capturar más compradores)
     - Si el costo del envío es ALTO  → subir umbral (mayor precisión, menos desperdicio)
 
  4. ROI estimado: con AUC {auc:.2f}, el modelo puede reducir campañas
     mal dirigidas hasta en un ~40%, optimizando los $80M COP mensuales.
""")

In [ ]:
# 9. ANÁLISIS CANAL × HORA × DESCUENTO (negocio) 

print("=" * 55)
print("  TOP 5 COMBINACIONES CANAL + HORA (mayor conversión)")
print("=" * 55)
combo = (df.groupby(["canal_marketing", "hora_envio"])
         .agg(tasa_conversion=("realizo_compra", "mean"),
              n_campanas=("realizo_compra", "count"))
         .assign(tasa_conversion=lambda x: (x["tasa_conversion"] * 100).round(1))
         .sort_values("tasa_conversion", ascending=False)
         .head(5))
print(combo.to_string())
 
print("\n" + "=" * 55)
print("  CONVERSIÓN POR NIVEL DE DESCUENTO (tramos)")
print("=" * 55)
df["tramo_descuento"] = pd.cut(df["descuento_porcentaje"],
                                bins=[-1, 0, 10, 20, 30],
                                labels=["Sin descuento", "1–10%", "11–20%", "21–30%"])
desc_conv = (df.groupby("tramo_descuento", observed=True)["realizo_compra"]
             .mean() * 100).round(1)
print(desc_conv.to_string())
print()
print("  [!] Si el descuento alto no mueve la aguja de conversión,")
print("      hay espacio para reducirlo y mejorar el margen.")
 
print("\n" + "=" * 55)
print("  CÓDIGO FINALIZADO – todos los artefactos guardados")
print("=" * 55)
 